# Error Compensation (2D theorem-regime)

Source: `C:\\Users\\Xing\\Desktop\\0508 errorcompensation.md`.

This notebook runs only the minimal deterministic 2D experiments in the theorem regime, with correct BECR separation (`rho_t` vs `s_t`).

Outputs:
- figures -> `figures/`
- JSON summaries -> `results/`


In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE_DIR = Path("C:/Users/Xing/Desktop/error_compensation_2d_20260508_110807")
FIG_DIR = BASE_DIR / "figures"
RES_DIR = BASE_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


BASE_DIR: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807


In [2]:
@dataclass
class Quad2D:
    a: float
    b: float

    def value(self, x: np.ndarray) -> float:
        return 0.5 * (self.a * float(x[0] ** 2) + self.b * float(x[1] ** 2))

    def grad(self, x: np.ndarray) -> np.ndarray:
        return np.array([self.a * float(x[0]), self.b * float(x[1])], dtype=float)


def unit(theta: float) -> np.ndarray:
    return np.array([np.cos(theta), np.sin(theta)], dtype=float)


def project(g: np.ndarray, p: np.ndarray) -> np.ndarray:
    return p * float(np.dot(p, g))


In [3]:
def phi_positive_eps(r_abs: float, eps_a: float, eps_s: float) -> float:
    return float(r_abs) / ((float(r_abs) + float(eps_a)) * (float(r_abs) + float(eps_s)))


def theorem_checks(*, a: float, b: float, lr: float, eps_a: float, eps_s: float, x0: np.ndarray) -> dict:
    R = float(a) * abs(float(x0[0]))
    r_star = float(np.sqrt(float(eps_a) * float(eps_s)))
    r0 = min(R, r_star)
    M0 = phi_positive_eps(r0, eps_a, eps_s)

    ok1 = (lr * a) < (2.0 * eps_a)
    ok2 = (lr * b * M0) <= 0.5

    info = {
        "eta_a": lr * a,
        "two_eps_a": 2.0 * eps_a,
        "ok_eta_a_lt_2epsa": bool(ok1),
        "R": R,
        "r_star": r_star,
        "r0": r0,
        "M0": M0,
        "eta_b_M0": lr * b * M0,
        "ok_eta_b_M0_le_half": bool(ok2),
    }
    print("Theorem checks")
    print("--------------")
    print("eta*a < 2 eps_a:", info["eta_a"], "<", info["two_eps_a"], "=", info["ok_eta_a_lt_2epsa"])
    print("M0:", info["M0"])
    print("eta*b*M0 <= 1/2:", info["eta_b_M0"], "<=", 0.5, "=", info["ok_eta_b_M0_le_half"])
    return info


In [4]:
class Optim:
    name: str

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        raise NotImplementedError


class ProjSGD(Optim):
    def __init__(self, *, lr: float, angle_fn):
        self.name = "proj_sgd"
        self.lr = float(lr)
        self.angle_fn = angle_fn

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        u = project(g, p)
        x_next = x - self.lr * u
        return x_next, {
            "theta": float(self.angle_fn(t)),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
        }


class FiraScalar(Optim):
    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, s_clip: tuple[float, float] | None = None):
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.s_clip = s_clip
        self.name = "fira_raw" if s_clip is None else "fira_clipped"

    def _psi(self, r: float) -> float:
        # Toy Adam-like mapping used by the theorem-regime counterexample.
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        if self.s_clip is None:
            s_t = float(s_raw)
        else:
            s_min, s_max = self.s_clip
            s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        u_perp = s_t * g_perp
        u = u_parallel + u_perp
        x_next = x - self.lr * u

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float("nan"),
        }


class BECRFiraToy(Optim):
    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, rho: float, s_clip: tuple[float, float], limiter_gamma: float | None = None):
        self.name = "becr_fira_toy"
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.rho = float(rho)
        self.s_clip = s_clip
        self.limiter_gamma = limiter_gamma

        self.e = np.zeros((2,), dtype=float)  # raw-gradient units
        self._prev_u_perp_norm = None

    def _psi(self, r: float) -> float:
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        # This is the toy recovery scale s_t (separate from rho).
        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        s_min, s_max = self.s_clip
        s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        # BECR raw-gradient residual compensation.
        h = g_perp + self.e  # raw-gradient units
        z = self.rho * h
        z_eff = z.copy()
        u_perp = s_t * z_eff

        if self.limiter_gamma is not None:
            cur = float(np.linalg.norm(u_perp))
            if (
                self._prev_u_perp_norm is not None
                and self._prev_u_perp_norm > 0
                and cur > self.limiter_gamma * self._prev_u_perp_norm
            ):
                tau = self.limiter_gamma * self._prev_u_perp_norm / cur
                z_eff = tau * z
                u_perp = s_t * z_eff
                cur = float(np.linalg.norm(u_perp))
            self._prev_u_perp_norm = cur

        # Residual update must use the effective transmitted raw signal.
        self.e = h - z_eff

        u = u_parallel + u_perp
        x_next = x - self.lr * u

        gperp_norm = float(np.linalg.norm(g_perp))
        phi_eff = float(np.linalg.norm(u_perp) / (gperp_norm + 1e-12))

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "rho": float(self.rho),
            "phi_eff": float(phi_eff),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float(np.linalg.norm(self.e)),
        }


In [5]:
class WrongUnitsNaiveEF(Optim):
    """Ablation: stores residual in update units (includes lr), which is NOT BECR.

This is intentionally wrong and included only to demonstrate why BECR must keep e_t in raw-gradient units.
    """

    def __init__(self, *, lr: float, angle_fn, eps_a: float, eps_s: float, s_clip: tuple[float, float], rho: float):
        self.name = "wrong_units_naive_ef"
        self.lr = float(lr)
        self.angle_fn = angle_fn
        self.eps_a = float(eps_a)
        self.eps_s = float(eps_s)
        self.s_clip = s_clip
        self.rho = float(rho)

        self.e = np.zeros((2,), dtype=float)  # update-units (wrong)

    def _psi(self, r: float) -> float:
        return float(r) / (abs(float(r)) + self.eps_a)

    def step(self, x: np.ndarray, g: np.ndarray, t: int) -> tuple[np.ndarray, dict]:
        p = unit(float(self.angle_fn(t)))
        r = float(np.dot(p, g))
        g_parallel = p * r
        g_perp = g - g_parallel

        psi_r = float(self._psi(r))
        u_parallel = p * psi_r

        s_raw = abs(psi_r) / (abs(r) + self.eps_s)
        s_min, s_max = self.s_clip
        s_t = float(np.clip(float(s_raw), float(s_min), float(s_max)))

        # Wrong: mixes learning-rate-scaled residual with raw-gradient signal.
        h = self.e + self.lr * g_perp
        z = self.rho * h
        self.e = h - z

        # Wrong: z is in update units, then multiplied by s_t and again by lr below.
        u_perp = s_t * z
        u = u_parallel + u_perp
        x_next = x - self.lr * u

        return x_next, {
            "theta": float(self.angle_fn(t)),
            "phi_raw": float(s_raw),
            "s": float(s_t),
            "rho": float(self.rho),
            "x": float(x_next[0]),
            "y": float(x_next[1]),
            "e_norm": float(np.linalg.norm(self.e)),
        }


In [6]:
def run(*, exp_name: str, quad: Quad2D, angle_fn, optimizers: list[Optim], x0: np.ndarray, steps: int) -> dict:
    traces = {}

    for opt in optimizers:
        x = np.array(x0, dtype=float)
        xs = np.zeros((steps + 1,), dtype=float)
        ys = np.zeros((steps + 1,), dtype=float)
        f = np.zeros((steps + 1,), dtype=float)
        grad_norm = np.zeros((steps + 1,), dtype=float)
        phi_raw = np.zeros((steps + 1,), dtype=float)
        s = np.zeros((steps + 1,), dtype=float)
        e_norm = np.zeros((steps + 1,), dtype=float)
        theta = np.zeros((steps + 1,), dtype=float)

        xs[0], ys[0] = float(x[0]), float(x[1])
        f[0] = quad.value(x)
        grad_norm[0] = float(np.linalg.norm(quad.grad(x)))
        phi_raw[0] = float("nan")
        s[0] = float("nan")
        e_norm[0] = float("nan")
        theta[0] = float(angle_fn(0))

        for t in range(steps):
            g = quad.grad(x)
            x, info = opt.step(x, g, t)

            xs[t + 1], ys[t + 1] = float(info["x"]), float(info["y"])
            f[t + 1] = quad.value(x)
            grad_norm[t + 1] = float(np.linalg.norm(quad.grad(x)))
            phi_raw[t + 1] = float(info.get("phi_raw", float("nan")))
            s[t + 1] = float(info.get("s", float("nan")))
            e_norm[t + 1] = float(info.get("e_norm", float("nan")))
            theta[t + 1] = float(info.get("theta", float(angle_fn(t))))

        traces[opt.name] = {
            "x": xs,
            "y": ys,
            "f": f,
            "grad_norm": grad_norm,
            "phi_raw": phi_raw,
            "s": s,
            "phi_cum": np.nancumsum(phi_raw),
            "s_cum": np.nancumsum(s),
            "e_norm": e_norm,
            "theta": theta,
        }

    def _plot_series(keys: list[str], ylabel: str, out_name: str, *, yscale: str | None = None):
        fig, ax = plt.subplots(figsize=(6.2, 4.0))
        for name, tr in traces.items():
            for k in keys:
                ax.plot(tr[k], linewidth=1.5, label=f"{name}:{k}")
        ax.set_title(f"{exp_name}: {out_name}")
        ax.set_xlabel("step")
        ax.set_ylabel(ylabel)
        if yscale is not None:
            ax.set_yscale(yscale)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{exp_name}__{out_name}.png", dpi=180)
        plt.close(fig)

    # Required plots.
    _plot_series(["x"], "x_t", "x")
    _plot_series(["y"], "y_t", "y")
    _plot_series(["phi_raw"], "phi_raw", "phi_raw")
    _plot_series(["s"], "s_t", "s")
    _plot_series(["phi_cum"], "cum(phi_raw)", "phi_cum")
    _plot_series(["s_cum"], "cum(s)", "s_cum")
    _plot_series(["e_norm"], "||e_t||", "e_norm", yscale=None)
    _plot_series(["grad_norm"], "||grad||", "grad_norm", yscale="log")
    _plot_series(["f"], "f(x_t)", "objective", yscale="log")

    # Save raw traces for reuse (paper figures, ablations).
    npz_payload = {}
    for name, tr in traces.items():
        for k, v in tr.items():
            npz_payload[f"{name}__{k}"] = np.asarray(v)
    np.savez_compressed(RES_DIR / f"{exp_name}__traces.npz", **npz_payload)

    # Paper-ready composite figure.
    label_map = {
        "proj_sgd": "ProjSGD",
        "fira_raw": "Fira (raw)",
        "fira_clipped": "Fira (clipped)",
        "becr_fira_toy": "BECR-Fira",
        "wrong_units_naive_ef": "Naive EF (wrong units)",
    }
    color_map = {
        "proj_sgd": "#666666",
        "fira_raw": "#d62728",
        "fira_clipped": "#1f77b4",
        "becr_fira_toy": "#2ca02c",
        "wrong_units_naive_ef": "#ff7f0e",
    }
    order = ["proj_sgd", "fira_raw", "fira_clipped", "becr_fira_toy", "wrong_units_naive_ef"]

    fig, axs = plt.subplots(2, 2, figsize=(10.2, 7.2))
    ax_y, ax_s, ax_scum, ax_e = axs[0, 0], axs[0, 1], axs[1, 0], axs[1, 1]

    # (a) y_t
    min_pos_y = None
    max_y = 0.0
    for name in order:
        if name not in traces:
            continue
        y = traces[name]["y"]
        max_y = max(max_y, float(np.nanmax(y)))
        pos = y[np.isfinite(y) & (y > 0)]
        if pos.size:
            v = float(np.min(pos))
            min_pos_y = v if min_pos_y is None else min(min_pos_y, v)
        ax_y.plot(y, linewidth=1.8, color=color_map.get(name, None), label=label_map.get(name, name))
    if min_pos_y is not None:
        ax_y.set_yscale("log")
        ax_y.set_ylim(min_pos_y * 0.8, max_y * 1.2)
    ax_y.set_title("(a) y_t")
    ax_y.set_xlabel("step")
    ax_y.set_ylabel("y")
    ax_y.grid(True, alpha=0.3)
    ax_y.legend(loc="best", fontsize=9)

    # (b) applied orthogonal scale s_t
    min_pos_s = None
    max_s = 0.0
    for name in order:
        if name not in traces:
            continue
        s = traces[name]["s"]
        pos = s[np.isfinite(s) & (s > 0)]
        if not pos.size:
            continue
        max_s = max(max_s, float(np.max(pos)))
        v = float(np.min(pos))
        min_pos_s = v if min_pos_s is None else min(min_pos_s, v)
        ax_s.plot(s, linewidth=1.6, color=color_map.get(name, None))
    if min_pos_s is not None:
        ax_s.set_yscale("log")
        ax_s.set_ylim(min_pos_s * 0.8, max_s * 1.2)
    ax_s.set_title("(b) applied scale s_t")
    ax_s.set_xlabel("step")
    ax_s.set_ylabel("s")
    ax_s.grid(True, alpha=0.3)

    # (c) cumulative applied scale
    for name in order:
        if name not in traces:
            continue
        sc = traces[name]["s_cum"]
        if not np.isfinite(sc).any():
            continue
        ax_scum.plot(sc, linewidth=1.6, color=color_map.get(name, None))
    ax_scum.set_title("(c) cumulative scale sum_t s_t")
    ax_scum.set_xlabel("step")
    ax_scum.set_ylabel("sum s")
    ax_scum.grid(True, alpha=0.3)

    # (d) residual norm (only for BECR / ablation)
    min_pos_e = None
    max_e = 0.0
    for name in order:
        if name not in traces:
            continue
        en = traces[name]["e_norm"]
        pos = en[np.isfinite(en) & (en > 0)]
        if not pos.size:
            continue
        max_e = max(max_e, float(np.max(pos)))
        v = float(np.min(pos))
        min_pos_e = v if min_pos_e is None else min(min_pos_e, v)
        ax_e.plot(en, linewidth=1.6, color=color_map.get(name, None))
    if min_pos_e is not None:
        ax_e.set_yscale("log")
        ax_e.set_ylim(min_pos_e * 0.8, max_e * 1.2)
    ax_e.set_title("(d) residual norm ||e_t||")
    ax_e.set_xlabel("step")
    ax_e.set_ylabel("||e||")
    ax_e.grid(True, alpha=0.3)

    fig.suptitle(exp_name)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(FIG_DIR / f"{exp_name}__PAPER_MAIN.png", dpi=220)
    fig.savefig(FIG_DIR / f"{exp_name}__PAPER_MAIN.pdf")
    plt.close(fig)

    # Summary table.
    summary = {
        "exp_name": exp_name,
        "quad": {"a": quad.a, "b": quad.b},
        "steps": int(steps),
        "optimizers": {},
    }

    for name, tr in traces.items():
        summary["optimizers"][name] = {
            "x_final": float(tr["x"][-1]),
            "y_final": float(tr["y"][-1]),
            "grad_norm_final": float(tr["grad_norm"][-1]),
            "residual_norm_final": float(tr["e_norm"][-1]) if np.isfinite(tr["e_norm"][-1]) else None,
            "sum_phi_raw": float(tr["phi_cum"][-1]) if np.isfinite(tr["phi_cum"][-1]) else None,
            "sum_s": float(tr["s_cum"][-1]) if np.isfinite(tr["s_cum"][-1]) else None,
        }

    (RES_DIR / f"{exp_name}__summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


In [7]:
# Theorem-regime parameters (from 0508 note)
a = 1.0
b = 1.0
lr = 0.5
eps_a = 1.0
eps_s = 1.0
x0 = np.array([1.0, 1.0], dtype=float)

quad = Quad2D(a=a, b=b)

# Fixed bad subspace P=e1.
angle_fn = lambda t: 0.0

checks = theorem_checks(a=a, b=b, lr=lr, eps_a=eps_a, eps_s=eps_s, x0=x0)
(RES_DIR / "THEOREM_CHECKS.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")

opts = [
    ProjSGD(lr=lr, angle_fn=angle_fn),
    FiraScalar(lr=lr, angle_fn=angle_fn, eps_a=eps_a, eps_s=eps_s, s_clip=None),
    FiraScalar(lr=lr, angle_fn=angle_fn, eps_a=eps_a, eps_s=eps_s, s_clip=(0.2, 0.5)),
    BECRFiraToy(lr=lr, angle_fn=angle_fn, eps_a=eps_a, eps_s=eps_s, rho=0.5, s_clip=(0.2, 0.5), limiter_gamma=None),
    WrongUnitsNaiveEF(lr=lr, angle_fn=angle_fn, eps_a=eps_a, eps_s=eps_s, s_clip=(0.2, 0.5), rho=0.5),
]

summary = run(
    exp_name="theorem_regime_fixed_subspace",
    quad=quad,
    angle_fn=angle_fn,
    optimizers=opts,
    x0=x0,
    steps=600,
)

(RES_DIR / "ALL_SUMMARIES.json").write_text(json.dumps([summary], indent=2), encoding="utf-8")
print("Wrote figures to:", FIG_DIR)
print("Wrote results to:", RES_DIR)


Theorem checks
--------------
eta*a < 2 eps_a: 0.5 < 2.0 = True
M0: 0.25
eta*b*M0 <= 1/2: 0.125 <= 0.5 = True


Wrote figures to: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807\figures
Wrote results to: C:\Users\Xing\Desktop\error_compensation_2d_20260508_110807\results
